In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

# Day 05 · Tabular Models Only

Notebook único de evidencia para la línea tabular pura post-fix.

Objetivos:
- comparar `CatBoost`, `LightGBM` y `XGBoost` contra el baseline corregido `LR_smote_0.5`;
- comparar cada tabular contra su LR equivalente por dataset para testar hipótesis de no linealidad;
- decidir qué parejas pasan a tuning, balanceo adicional o descarte;
- mantener Brent fuera de este notebook.

## Contexto y contrato fijo

- `cutoff_date`: `2028-02-21`
- holdout oficial: mismo universo post-Day 04 rerun
- métricas oficiales: `accuracy`, `balanced_accuracy`, `f1_pos`, `top1_hit`, `top2_hit`, `coverage`
- gate oficial de modelo puro: `top2/bal_acc/coverage`
- comparativa secundaria con policy Day 04: solo para el mejor tabular puro si queda en zona defendible

In [2]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "artifacts").exists() and (PROJECT_ROOT.parent / "artifacts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DAY05_ROOT = PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05"
RUN_SUMMARIES = sorted(DAY05_ROOT.glob("*_run_summary.json"), key=lambda path: path.stat().st_mtime)
if not RUN_SUMMARIES:
    raise FileNotFoundError("No hay run summaries Day 05 todavía. Ejecuta el runner público antes de usar el notebook.")

LATEST_SUMMARY_PATH = RUN_SUMMARIES[-1]
LATEST_SUMMARY = json.loads(LATEST_SUMMARY_PATH.read_text(encoding="utf-8"))
RUN_ID = LATEST_SUMMARY["run_id"]
CANONICAL_CSV = DAY05_ROOT / f"{RUN_ID}_canonical_candidates.csv"
SELECTION_JSON = DAY05_ROOT / f"{RUN_ID}_selection_decisions.json"
POLICY_JSON = DAY05_ROOT / f"{RUN_ID}_policy_summary.json"
canonical_df = pd.read_csv(CANONICAL_CSV)
selection_payload = json.loads(SELECTION_JSON.read_text(encoding="utf-8"))
policy_payload = json.loads(POLICY_JSON.read_text(encoding="utf-8")) if POLICY_JSON.exists() else {}
RUN_ID, LATEST_SUMMARY_PATH.name, canonical_df.shape

('20260306T_day05_run01', '20260306T_day05_run01_run_summary.json', (15, 35))

## Evidencia principal vs baseline corregido

La lectura principal del día es siempre `modelo puro vs baseline corregido`.

In [3]:
main_cols = [
    'dataset_alias', 'model_family', 'model_variant', 'variant_stage', 'balance_tag',
    'top2_hit', 'balanced_accuracy', 'f1_pos', 'coverage',
    'delta_top2_vs_baseline', 'delta_bal_acc_vs_baseline', 'delta_coverage_vs_baseline',
    'gate_pass', 'promotion_decision', 'eligible_primary_phase2', 'selected_for_phase2', 'selected_for_phase3'
]
canonical_df[main_cols].sort_values(
    ['delta_top2_vs_baseline', 'delta_bal_acc_vs_baseline', 'top2_hit', 'balanced_accuracy'],
    ascending=False,
).reset_index(drop=True)

,dataset_alias,model_family,model_variant,variant_stage,balance_tag,top2_hit,balanced_accuracy,f1_pos,coverage,delta_top2_vs_baseline,delta_bal_acc_vs_baseline,delta_coverage_vs_baseline,gate_pass,promotion_decision,eligible_primary_phase2,selected_for_phase2,selected_for_phase3
0,V2_TRANSPORT_CARRY30D_ONLY,XGBOOST,V2_TRANSPORT_CARRY30D_ONLY_XGBOOST_v1,phase1,NaN,0.884922,0.752748,0.632084,1.0,2.658614e-02,-0.112336,0.0,False,keep_baseline,False,False,False
1,V2_DISPERSION,XGBOOST,V2_DISPERSION_XGBOOST_v1,phase1,NaN,0.884922,0.721564,0.582333,1.0,2.658614e-02,-0.143520,0.0,False,keep_baseline,False,False,False
2,V2_TRANSPORT_CARRY30D_ONLY,LIGHTGBM,V2_TRANSPORT_CARRY30D_ONLY_LIGHTGBM_v1,phase1,NaN,0.883023,0.748444,0.627923,1.0,2.468717e-02,-0.116640,0.0,False,keep_baseline,False,False,False
3,V2_TRANSPORT_ONLY,XGBOOST,V2_TRANSPORT_ONLY_XGBOOST_v1,phase1,NaN,0.882264,0.737817,0.611937,1.0,2.392758e-02,-0.127267,0.0,False,keep_baseline,False,False,False
4,V2_TRANSPORT_ONLY,LIGHTGBM,V2_TRANSPORT_ONLY_LIGHTGBM_v1,phase1,NaN,0.880744,0.739570,0.611759,1.0,2.240840e-02,-0.125514,0.0,False,keep_baseline,False,False,False
5,V2,XGBOOST,V2_XGBOOST_v1,phase1,NaN,0.874288,0.770684,0.653491,1.0,1.595188e-02,-0.094400,0.0,False,keep_baseline,False,False,False
6,V2,LIGHTGBM,V2_LIGHTGBM_v1,phase1,NaN,0.871629,0.743546,0.623236,1.0,1.329332e-02,-0.121538,0.0,False,keep_baseline,False,False,False
7,V2_DISPERSION,LIGHTGBM,V2_DISPERSION_LIGHTGBM_v1,phase1,NaN,0.868591,0.715301,0.572261,1.0,1.025496e-02,-0.149783,0.0,False,keep_baseline,False,False,False
8,V2_COMPETITION,LIGHTGBM,V2_COMPETITION_LIGHTGBM_v1,phase1,NaN,0.865553,0.750865,0.633875,1.0,7.216602e-03,-0.114219,0.0,False,keep_baseline,False,False,False
9,V2_TRANSPORT_CARRY30D_ONLY,CATBOOST,V2_TRANSPORT_CARRY30D_ONLY_CATBOOST_v1,phase1,NaN,0.863654,0.751875,0.628871,1.0,5.317627e-03,-0.113209,0.0,False,keep_baseline,False,False,False


## Evidencia secundaria vs LR equivalente

Esta tabla responde a la hipótesis de Day 05: si una familia de modelo no lineal está capturando señal que el LR equivalente del mismo dataset no captó.

In [4]:
secondary_cols = [
    'dataset_alias', 'lr_equivalent_variant', 'model_family', 'model_variant', 'variant_stage',
    'top2_hit', 'balanced_accuracy', 'coverage',
    'delta_top2_vs_lr_equivalente', 'delta_bal_acc_vs_lr_equivalente', 'delta_coverage_vs_lr_equivalente',
    'eligible_secondary_phase2'
]
canonical_df[secondary_cols].sort_values(
    ['delta_top2_vs_lr_equivalente', 'delta_bal_acc_vs_lr_equivalente', 'top2_hit'],
    ascending=False,
).reset_index(drop=True)

,dataset_alias,lr_equivalent_variant,model_family,model_variant,variant_stage,top2_hit,balanced_accuracy,coverage,delta_top2_vs_lr_equivalente,delta_bal_acc_vs_lr_equivalente,delta_coverage_vs_lr_equivalente,eligible_secondary_phase2
0,V2_DISPERSION,V2_DISPERSION_LR_smote_0.5_v1,XGBOOST,V2_DISPERSION_XGBOOST_v1,phase1,0.884922,0.721564,1.0,0.027725,-0.143493,0.0,False
1,V2_TRANSPORT_CARRY30D_ONLY,V2_TRANSPORT_CARRY30D_ONLY_LR_smote_0.5_v1,XGBOOST,V2_TRANSPORT_CARRY30D_ONLY_XGBOOST_v1,phase1,0.884922,0.752748,1.0,0.022408,-0.111333,0.0,False
2,V2_TRANSPORT_ONLY,V2_TRANSPORT_ONLY_LR_smote_0.5_v1,XGBOOST,V2_TRANSPORT_ONLY_XGBOOST_v1,phase1,0.882264,0.737817,1.0,0.022029,-0.128030,0.0,False
3,V2_TRANSPORT_ONLY,V2_TRANSPORT_ONLY_LR_smote_0.5_v1,LIGHTGBM,V2_TRANSPORT_ONLY_LIGHTGBM_v1,phase1,0.880744,0.739570,1.0,0.020509,-0.126277,0.0,False
4,V2_TRANSPORT_CARRY30D_ONLY,V2_TRANSPORT_CARRY30D_ONLY_LR_smote_0.5_v1,LIGHTGBM,V2_TRANSPORT_CARRY30D_ONLY_LIGHTGBM_v1,phase1,0.883023,0.748444,1.0,0.020509,-0.115637,0.0,False
5,V2,LR_smote_0.5,XGBOOST,V2_XGBOOST_v1,phase1,0.874288,0.770684,1.0,0.015952,-0.094400,0.0,False
6,V2,LR_smote_0.5,LIGHTGBM,V2_LIGHTGBM_v1,phase1,0.871629,0.743546,1.0,0.013293,-0.121538,0.0,False
7,V2_COMPETITION,V2_COMPETITION_LR_smote_0.5_v1,LIGHTGBM,V2_COMPETITION_LIGHTGBM_v1,phase1,0.865553,0.750865,1.0,0.013293,-0.116560,0.0,False
8,V2_DISPERSION,V2_DISPERSION_LR_smote_0.5_v1,LIGHTGBM,V2_DISPERSION_LIGHTGBM_v1,phase1,0.868591,0.715301,1.0,0.011394,-0.149756,0.0,False
9,V2_COMPETITION,V2_COMPETITION_LR_smote_0.5_v1,XGBOOST,V2_COMPETITION_XGBOOST_v1,phase1,0.861755,0.757905,1.0,0.009495,-0.109520,0.0,False


## Decisiones y trade-offs

- `selection_payload` recoge qué parejas pasaron a Fase 2 y qué variante pura queda mejor situada.
- Los search logs de Fase 2/Fase 3 viven fuera del registry oficial.
- La policy Day 04 solo se consulta una vez y solo si el mejor tabular puro queda en zona defendible.

In [5]:
selection_payload

{'run_id': '20260306T_day05_run01',
 'phase2_selected_variants': [],
 'phase3_selected_variants': [],
 'best_final_pure_variant': 'V2_TRANSPORT_CARRY30D_ONLY_XGBOOST_v1'}

In [6]:
policy_payload

{'run_id': '20260306T_day05_run01',
 'executed': False,
 'reason': 'best_pure_not_policy_eligible',
 'best_final_pure_variant': 'V2_TRANSPORT_CARRY30D_ONLY_XGBOOST_v1'}

## Decisión de cierre Day 05

Checklist final a dejar respondido tras la ejecución:
- qué candidato, si alguno, merece `promote` como modelo puro;
- qué hipótesis de no linealidad quedan respaldadas por evidencia;
- qué parejas se descartan definitivamente;
- si procede o no abrir Day 05.5 desde baseline corregido o desde el mejor tabular puro.